In [1]:
import torch
import numpy as np
from data.data_utils import get_dataloaders
from models import get_model
from sde import configure_sde
import torchvision.utils as vutils
import matplotlib.pyplot as plt
%matplotlib inline
from configs import load_config
from utils.sampling_utils import generate_samples, plot_samples

In [2]:
def load_model(model, checkpoint_path, device=None):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    if 'model_state_dict' not in checkpoint:
        raise KeyError("Checkpoint does not contain 'model_state_dict'")
    model.load_state_dict(checkpoint['model_state_dict'])
    return model

In [3]:
def get_generation_callback(vis_callback, G=None, guidance_interval = None):
    if vis_callback == 'base':
        def generation_callback(batch, sde, diffusion_model, steps, shape, device):
            #Unconditional diffusion models
            _, y = batch
            #y is the condition. 
            samples = generate_samples(y, sde, diffusion_model, steps, shape, device, G=G, guidance_interval=guidance_interval) 
            return samples
    
    elif vis_callback == 'scoreVAE':
        def generation_callback(batch, sde, diffusion_model, steps, shape, device):
            x, y = batch
            z = diffusion_model.encode(x)  # Get the latent representation
            cond = [y, z]
            reconstruction = generate_samples(cond, sde, diffusion_model, steps, shape, device, G=G, guidance_interval=guidance_interval)
            return reconstruction
    else:
        raise ValueError(f"Unknown vis_callback '{vis_callback}'. Expected 'base' or 'scoreVAE'.")
    return generation_callback

In [4]:
config_path = 'configs/cifar10/scoreVAE.py'
config = load_config(config_path)
device = config.training.device
train_loader, val_loader, test_loader = get_dataloaders(config.data)
num_samples = config.training.num_samples
shape = (num_samples, *config.data.shape)

In [5]:
def sample_with_guidance(
    model, sde, dataloader, steps, shape, device, 
    get_generation_callback, 
    vis_callback,  # pass this in explicitly
    guidance_weight, 
    guidance_interval, 
    num_batches=1,
    plot_grid=True,
    grid_save_path=None
):
    """
    Generate reconstructions for a single guidance weight/interval, compute mean L2 loss.
    For the first batch, plot a grid of reconstructions vs originals.
    Returns the mean L2 loss and all per-sample losses.
    """
    model.eval()
    generation_callback = get_generation_callback(
        vis_callback,
        # G=guidance_weight, 
        # guidance_interval=guidance_interval
        G=None, 
        guidance_interval=None
    )

    l2_losses = []
    plotted = False
    with torch.no_grad():
        for i, data in enumerate(dataloader):
            x, y = data if isinstance(data, (tuple, list)) else (data, None)
            x = x.to(device)
            if y is not None:
                y = y.to(device)
            recon = generation_callback(
                batch = (x, y), 
                sde = sde, 
                diffusion_model = model, 
                steps = steps, 
                shape = shape, 
                device = device,
                # G=guidance_weight,
                # guidance_interval=guidance_interval
                )
            l2 = torch.mean((recon - x) ** 2, dim=tuple(range(1, recon.ndim)))
            l2_losses.extend(l2.cpu().numpy())

            # Plot grid for the first batch
            if plot_grid and not plotted:
                # Clamp images to [0, 1] for visualization
                orig = x.clone().detach().cpu()
                rec = recon.clone().detach().cpu()
                # orig = orig.clamp(0, 1)
                # rec = rec.clamp(0, 1)
                # Make grids
                nrow = int(np.sqrt(orig.shape[0]))
                grid_orig = vutils.make_grid(orig, nrow=nrow, normalize=True)
                grid_recon = vutils.make_grid(rec, nrow=nrow, normalize=True)
                # Plot
                fig, axs = plt.subplots(1, 2, figsize=(10, 5))
                axs[0].imshow(np.transpose(grid_orig.numpy(), (1, 2, 0)))
                axs[0].set_title("Originals")
                axs[0].axis("off")
                axs[1].imshow(np.transpose(grid_recon.numpy(), (1, 2, 0)))
                axs[1].set_title("Reconstructions")
                axs[1].axis("off")
                plt.tight_layout()
                if grid_save_path is not None:
                    plt.savefig(grid_save_path, bbox_inches='tight')
                    print(f"Grid plot saved to {grid_save_path}")
                plt.show()
                plotted = True

            if i + 1 >= num_batches:
                break
    mean_l2 = float(np.nanmean(l2_losses))
    return mean_l2, l2_losses

def grid_guided_sampling(
    model, sde, dataloader, steps, shape, device, 
    get_generation_callback,
    vis_callback,        # pass this in explicitly
    guidance_weights, 
    guidance_intervals,
    num_batches=1,
):
    """
    For all pairs of weights and intervals, call sample_with_guidance. Returns a results dict.
    """
    results = {}
    for w in guidance_weights:
        for interval in guidance_intervals:
            print(f"Sampling with weight={w}, interval={interval} ...")
            mean_l2, _ = sample_with_guidance(
                model, sde, dataloader, steps, shape, device, 
                get_generation_callback,
                vis_callback,
                w, interval, num_batches=num_batches,
            )
            results[(w, interval)] = mean_l2
            print(f"=> Mean L2 loss: {mean_l2:.6f}")
    return results

In [6]:
model = get_model(config.model).to(device)
ckpt_path = 'results/cifar10/scoreVAE_noise/checkpoints/Model_last.pth'
loaded_model = load_model(model=model, checkpoint_path = ckpt_path)

Model name from config: GuidedCombinedDiffusionEncoder
Importing module: models.GuidedCombinedDiffusionEncoder
Getting model class: GuidedCombinedDiffusionEncoder from module
Initializing model GuidedCombinedDiffusionEncoder with config: diffusion_model:
  attention_resolutions: !!python/tuple
  - 16
  attn_checkpoint: false
  channel_mult: !!python/tuple
  - 1
  - 2
  - 2
  - 2
  checkpoint: /home/rg625@ad.eng.cam.ac.uk/mnt/ScoreVAE/results/cifar10/unconditional/checkpoints/Model_epoch_390_loss_0.026.pth
  conv_resample: true
  dims: 2
  dropout: 0.1
  embed_channels: 512
  image_size: 32
  in_channels: 3
  input_channel_mult: null
  model_channels: 128
  network: BeatGANsUNet
  num_head_channels: -1
  num_heads: 1
  num_heads_upsample: -1
  num_input_res_blocks: null
  num_res_blocks: 4
  out_channels: 3
  resblock_updown: true
  resnet_cond_channels: null
  resnet_two_cond: false
  resnet_use_zero_module: true
  time_embed_channels: null
  use_checkpoint: false
  use_new_attention_o

In [7]:
mean_l2, l2_losses = sample_with_guidance(
    model=loaded_model,
    sde=configure_sde(config),
    dataloader=train_loader,
    steps=config.training.steps,
    shape=shape,
    device=device,
    get_generation_callback=get_generation_callback,
    vis_callback=config.training.vis_callback,
    guidance_weight=config.training.guidance_weight,
    guidance_interval=config.training.guidance_interval,
    num_batches=1,  # Evaluate just one batch
    plot_grid=True,
    grid_save_path="recon_grid.png"
)
print(f"Mean L2 loss: {mean_l2}")
print(f"L2 losses: {len(l2_losses)}")
print(f"L2 losses: {l2_losses}")

Min diffusion time: 4.0368668123846874e-05
Max diffusion time: 0.9335111379623413
Grid plot saved to recon_grid.png
Mean L2 loss: 0.008031435310840607
L2 losses: 128
L2 losses: [0.0023205183, 0.009448666, 0.003772142, 0.0073316693, 0.0026046548, 0.0036608027, 0.009842957, 0.010985362, 0.0039563635, 0.0044245645, 0.008849084, 0.01055191, 0.0053391894, 0.008601094, 0.0040665693, 0.006679966, 0.007564002, 0.0010125453, 0.0056803054, 0.01474086, 0.00789303, 0.015865464, 0.0087279035, 0.0046061063, 0.013030238, 0.009303922, 0.012959091, 0.014577454, 0.0024772063, 0.0037040736, 0.007609925, 0.0026359311, 0.010276752, 0.0154652735, 0.0059333183, 0.0059185354, 0.013417115, 0.018292552, 0.015050856, 0.007363518, 0.010746676, 0.005529873, 0.008264849, 0.0037210898, 0.0056883013, 0.009702787, 0.015555756, 0.0051777554, 0.017736878, 0.014212533, 0.004297233, 0.0054844394, 0.0048810323, 0.008295406, 0.015166385, 0.008445532, 0.006099885, 0.0073289177, 0.0025220336, 0.01815569, 0.009458159, 0.002747

In [8]:
guidance_weights=[0.1, 0.5, 0.8, 1.0, 1.25, 1.5, 2, 2.1, 2.5, 3, 4]
guidance_intervals=[(0.01, 0.99), (0.1, 0.9),(0.1, 0.8),(0.2, 0.9),(0.1, 0.5),]

In [1]:
import json
from dash import Dash, dcc, html, Output, Input
import plotly.graph_objs as go

# Load data
with open('ablation/grid_results.json', 'r') as f:
    data = json.load(f)

# Parse keys
lookup = {}
a_vals, b_vals, c_vals = set(), set(), set()

for key, val in data.items():
    a_b, c = key.split('-')
    a, b = map(float, a_b.split(','))
    c = float(c)
    lookup[(a, b, c)] = val
    a_vals.add(a)
    b_vals.add(b)
    c_vals.add(c)

a_vals = sorted(a_vals)
b_vals = sorted(b_vals)
c_vals = sorted(c_vals)

# Dash app
app = Dash(__name__)
app.title = "Ablation Grid Explorer"

app.layout = html.Div(style={"display": "flex", "padding": "40px"}, children=[
    html.Div(style={"width": "30%", "paddingRight": "20px"}, children=[
        html.H4("Select Parameters:"),
        html.Label("Weight (a):"),
        dcc.Slider(min=min(a_vals), max=max(a_vals), step=round((max(a_vals) - min(a_vals))/len(a_vals), 4),
                   marks={v: str(round(v, 2)) for v in a_vals}, value=a_vals[0], id="slider-a"),
        html.Br(),
        html.Label("Lower Limit (b):"),
        dcc.Slider(min=min(b_vals), max=max(b_vals), step=round((max(b_vals) - min(b_vals))/len(b_vals), 4),
                   marks={v: str(round(v, 2)) for v in b_vals}, value=b_vals[0], id="slider-b"),
        html.Br(),
        html.Label("Upper Limit (c):"),
        dcc.Slider(min=min(c_vals), max=max(c_vals), step=round((max(c_vals) - min(c_vals))/len(c_vals), 4),
                   marks={v: str(round(v, 2)) for v in c_vals}, value=c_vals[0], id="slider-c"),
    ]),

    html.Div(style={"width": "70%"}, children=[
        dcc.Graph(id="output-plot"),
        html.Div(id="output-text", style={"fontSize": "18px", "paddingTop": "10px"})
    ])
])

@app.callback(
    Output("output-plot", "figure"),
    Output("output-text", "children"),
    Input("slider-a", "value"),
    Input("slider-b", "value"),
    Input("slider-c", "value"),
)
def update_output(a, b, c):
    val = lookup.get((a, b, c))
    if val is not None:
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=["Loss"],
            y=[val],
            text=[f"{val:.4f}"],
            textposition="auto",
            marker_color="royalblue"
        ))
        fig.update_layout(title=f"Loss for a={a}, b={b}, c={c}",
                          yaxis_title="Loss",
                          height=400)
        return fig, f"Loss for configuration (a={a}, b={b}, c={c}): {val:.4f}"
    else:
        fig = go.Figure()
        fig.update_layout(title="No data available for this combination.",
                          height=400)
        return fig, "⚠️ No data for this combination."


In [9]:
# grid_guided_sampling(
#     model=loaded_model,
#     sde=configure_sde(config),
#     dataloader=test_loader,
#     steps=config.training.steps,
#     shape=shape,
#     device=device,
#     get_generation_callback=get_generation_callback,
#     vis_callback=config.training.vis_callback,
#     guidance_weights=guidance_weights,
#     guidance_intervals=guidance_intervals,
#     num_batches=1,  # Evaluate just one batch
# )
